# Redrob Hackathon — Pre-computation Notebook

**Run this ONCE in Google Colab (GPU runtime recommended)**

This notebook:
1. Mounts your Google Drive
2. Loads all 100,000 candidates
3. Builds rich text representation for each candidate
4. Embeds all candidates using SentenceTransformer
5. Builds a FAISS index
6. Saves `candidates.faiss` and `candidate_ids.json` to Drive

**After this notebook completes:**
- Download `candidates.faiss` and `candidate_ids.json` from Drive
- Place them in your local `redrob_ranker/artifacts/` folder
- Run `rank.py` locally to produce the submission CSV

**Runtime estimate:**
- With GPU (T4): ~8-12 minutes for 100k candidates
- With CPU: ~25-35 minutes

**Instructions:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install sentence-transformers faiss-gpu tqdm -q
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ⚠️  UPDATE THIS PATH to where your candidates.jsonl.gz is stored in Drive
CANDIDATES_PATH = '/content/drive/MyDrive/redrob/candidates.jsonl.gz'

# Output will be saved here
OUTPUT_DIR = '/content/drive/MyDrive/redrob/artifacts'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Drive mounted. Output will go to: {OUTPUT_DIR}')

In [ ]:
# ── Cell 3: Load all candidates ───────────────────────────────────────────
import gzip
import json
from tqdm import tqdm

print('Loading candidates...')
candidates = []

with gzip.open(CANDIDATES_PATH, 'rt', encoding='utf-8') as f:
    for line in tqdm(f, desc='Loading', unit=' candidates'):
        line = line.strip()
        if line:
            try:
                candidates.append(json.loads(line))
            except json.JSONDecodeError:
                pass

print(f'\n✅ Loaded {len(candidates):,} candidates')

# Sanity check
c = candidates[0]
print(f'Sample: {c["candidate_id"]} | {c["profile"]["current_title"]} | {c["profile"]["years_of_experience"]} yrs')

In [ ]:
# ── Cell 4: Build rich text for each candidate ────────────────────────────
# This is identical to src/text_builder.py but self-contained for Colab

CONSULTING_COMPANIES = {
    'tcs', 'tata consultancy', 'infosys', 'wipro', 'accenture',
    'cognizant', 'capgemini', 'hcl technologies', 'hcl', 'tech mahindra',
    'hexaware', 'mphasis', 'ltimindtree', 'mindtree', 'niit technologies',
    'syntel', 'zensar', 'igate', 'firstsource', 'wns global'
}

CORE_AI_SKILLS = {
    'embeddings', 'vector search', 'faiss', 'pinecone', 'weaviate', 'qdrant',
    'milvus', 'opensearch', 'elasticsearch', 'sentence-transformers',
    'dense retrieval', 'hybrid search', 'bm25', 'retrieval', 'ranking',
    'learning to rank', 'reranking', 'cross-encoder', 'bi-encoder',
    'pytorch', 'transformers', 'hugging face', 'huggingface',
    'llm', 'rag', 'fine-tuning', 'lora', 'qlora', 'peft',
    'nlp', 'natural language processing', 'text embeddings',
    'ndcg', 'mrr', 'map', 'python', 'machine learning', 'deep learning',
    'recommendation system', 'search system', 'information retrieval',
    'xgboost', 'lightgbm', 'neural ranking'
}

PROFICIENCY_WEIGHT = {'expert': 4, 'advanced': 3, 'intermediate': 2, 'beginner': 1}

def build_candidate_text(candidate):
    parts = []
    profile = candidate.get('profile', {})
    career = candidate.get('career_history', [])
    skills = candidate.get('skills', [])
    education = candidate.get('education', [])
    certifications = candidate.get('certifications', [])

    title = profile.get('current_title', '')
    headline = profile.get('headline', '')
    yoe = profile.get('years_of_experience', 0)
    location = profile.get('location', '')

    parts.append(f'Title: {title}. Experience: {yoe:.1f} years. Location: {location}.')
    if headline:
        parts.append(f'Headline: {headline}.')

    summary = profile.get('summary', '')
    if summary:
        parts.append(f'Summary: {summary}')

    career_parts = []
    for job in career:
        job_title = job.get('title', '')
        company = job.get('company', '')
        industry = job.get('industry', '')
        duration = job.get('duration_months', 0)
        description = job.get('description', '')
        company_size = job.get('company_size', '')
        is_current = job.get('is_current', False)
        is_consulting = any(cf in company.lower() for cf in CONSULTING_COMPANIES)
        company_type = 'consulting company' if is_consulting else 'product company'
        job_text = (
            f"{'Current' if is_current else 'Previous'}: {job_title} at {company} "
            f'({company_type}, {industry}, {company_size}, {duration}mo). {description}'
        )
        career_parts.append(job_text)

    if career_parts:
        parts.append('Career: ' + ' | '.join(career_parts))

    core_skills = []
    other_skills = []
    for skill in skills:
        name = skill.get('name', '')
        proficiency = skill.get('proficiency', 'beginner')
        duration = skill.get('duration_months', 0)
        is_core = any(kw in name.lower() for kw in CORE_AI_SKILLS)
        desc = f'{name} ({proficiency}, {duration/12:.1f}yr)'
        if is_core:
            core_skills.append(desc)
        else:
            other_skills.append(desc)

    if core_skills:
        parts.append(f'Key AI/ML skills: {", ".join(core_skills)}.')
    if other_skills:
        parts.append(f'Other skills: {", ".join(other_skills[:8])}.')

    for edu in education:
        field = edu.get('field_of_study', '')
        degree = edu.get('degree', '')
        institution = edu.get('institution', '')
        if field or degree:
            parts.append(f'Education: {degree} {field} {institution}.')

    if certifications:
        names = [c.get('name', '') for c in certifications[:4]]
        parts.append(f'Certifications: {", ".join(names)}.')

    return ' '.join(parts)


# Build texts for all candidates
print('Building candidate texts...')
texts = [build_candidate_text(c) for c in tqdm(candidates, desc='Building texts')]
print(f'\n✅ Built {len(texts):,} candidate texts')
print(f'Sample (first 200 chars):\n{texts[0][:200]}...')

In [ ]:
# ── Cell 5: Embed all candidates ──────────────────────────────────────────
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Load model
print('Loading SentenceTransformer model...')
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Embed all candidates in batches
# batch_size=512 for GPU, 128 for CPU
BATCH_SIZE = 512 if device == 'cuda' else 128

print(f'Encoding {len(texts):,} candidates (batch_size={BATCH_SIZE})...')
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,  # Normalize for cosine similarity via dot product
    convert_to_numpy=True
)

print(f'\n✅ Embeddings shape: {embeddings.shape}')
print(f'   dtype: {embeddings.dtype}')
print(f'   Memory: {embeddings.nbytes / 1024 / 1024:.1f} MB')

In [ ]:
# ── Cell 6: Build FAISS index ─────────────────────────────────────────────
import faiss

print('Building FAISS index...')
dimension = embeddings.shape[1]  # 384 for all-MiniLM-L6-v2

# IndexFlatIP = Inner Product (cosine similarity for normalized vectors)
# This is the most accurate index — no approximation.
# For 100k vectors at 384 dims, it fits easily in 16GB RAM.
index = faiss.IndexFlatIP(dimension)

# Add all embeddings to index
embeddings_f32 = embeddings.astype(np.float32)
index.add(embeddings_f32)

print(f'✅ FAISS index built')
print(f'   Total vectors: {index.ntotal:,}')
print(f'   Dimension: {dimension}')

# Quick sanity check — search for a test query
test_query = model.encode(
    ['Senior ML Engineer Python FAISS embeddings retrieval'],
    normalize_embeddings=True
).astype(np.float32)
D, I = index.search(test_query, 3)
print('\nSanity check — top 3 for test query:')
for dist, idx in zip(D[0], I[0]):
    c = candidates[idx]
    print(f'  [{dist:.4f}] {c["candidate_id"]} | {c["profile"]["current_title"]}')

In [ ]:
# ── Cell 7: Save artifacts to Drive ──────────────────────────────────────
import json

# Save FAISS index
faiss_path = f'{OUTPUT_DIR}/candidates.faiss'
faiss.write_index(index, faiss_path)
print(f'✅ FAISS index saved: {faiss_path}')

# Save ordered candidate IDs (must match embedding order exactly)
candidate_ids = [c['candidate_id'] for c in candidates]
ids_path = f'{OUTPUT_DIR}/candidate_ids.json'
with open(ids_path, 'w') as f:
    json.dump(candidate_ids, f)
print(f'✅ Candidate IDs saved: {ids_path}')

# Verify sizes
import os
faiss_size_mb = os.path.getsize(faiss_path) / 1024 / 1024
ids_size_mb = os.path.getsize(ids_path) / 1024 / 1024
print(f'\nFile sizes:')
print(f'  candidates.faiss: {faiss_size_mb:.1f} MB')
print(f'  candidate_ids.json: {ids_size_mb:.1f} MB')

print(f'\n✅ PRE-COMPUTATION COMPLETE!')
print(f'Download these two files from Drive and put them in artifacts/')
print(f'  {faiss_path}')
print(f'  {ids_path}')

In [ ]:
# ── Cell 8 (OPTIONAL): Download directly from Colab ──────────────────────
# If you want to download the files directly to your computer
# instead of going through Drive:

# from google.colab import files
# files.download(faiss_path)
# files.download(ids_path)
print('Uncomment the lines above to download directly')